# 01주차 실습: 빅데이터 문제 정의와 데이터 프로파일

강의: [PDF](../pdf/week01_bigdata_overview.pdf) · **PDF 실제 페이지 4–19, 27–34** (인쇄 번호와 다를 수 있음)

## 학습 목표
광고 이벤트의 품질과 규모를 측정하고 분석 문제 및 레이크하우스 구조를 정의한다.

권장 구성: 개념 확인 10분 → 코드 실행 30분 → 변경 실험 30분 → 결과 토의 20분.
설치는 README.md를 따릅니다. 새 커널에서 위에서 아래로 실행하세요.

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'labkit').is_dir()), None)
if root is None:
    candidate = here / '강의자료' / '실습코드'
    if candidate.is_dir(): root = candidate
if root is None:
    raise RuntimeError('실습코드 폴더에서 Jupyter를 시작하세요. README.md를 확인하세요.')
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labkit.common import ensure_data, DATA, output_dir, save_json
ensure_data()

## 데이터 규모와 품질

In [ ]:
df = pd.read_json(DATA / 'events.jsonl', lines=True)
profile = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'distinct': df.nunique()})
display(profile)
print('행:', len(df), '메모리 bytes:', df.memory_usage(deep=True).sum())
print('중복 event_id:', df.event_id.duplicated().sum())
assert df.event_id.is_unique
out = output_dir('week01')
profile.to_csv(out / 'profile.csv')

## 분포와 다중 검정

In [ ]:
rng = np.random.default_rng(42)
tail = rng.pareto(2, 10000) + 1
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(tail, bins=80); ax[0].set(xlim=(1, 15), title='Synthetic long tail')
pvalues = rng.uniform(size=(2000, 100))  # 귀무가설이 모두 참인 모의실험
rates = [(pvalues < .05).any(axis=1).mean(), (pvalues < .05 / 100).any(axis=1).mean()]
ax[1].bar(['Unadjusted', 'Bonferroni'], rates); ax[1].set_ylabel('Family-wise error rate')
plt.tight_layout(); plt.show()
save_json(out / 'problem_definition.json', {'question': '상품별 이벤트량과 매출성 value를 어떻게 집계할 것인가?',
 'unit': 'event', 'key': 'event_id', 'time': 'event_time UTC', 'layers': ['Landing', 'Bronze', 'Silver', 'Gold'],
 'note': '합성 데이터이며 value는 실제 매출을 뜻하지 않음'})

## 확장 과제 및 제출
5V 각각에 측정 지표를 하나씩 제안하세요. Landing→Bronze→Silver→Gold의 입력·출력을 그려 제출하고, 실제 매출 KPI에 필요한 추가 조건을 설명하세요.

제출: 실행한 노트북, 결과 표/그림, 변경한 조건과 해석. outputs/weekXX에 저장된 자료를 첨부하세요.

평가 기준: 개념 연결 25%, 실행·재현성 30%, 비교 실험 25%, 해석·한계 20%.